In [6]:
import numpy as np
import string

def string_to_one_hot(char_arrays):
    vocab = list(string.ascii_uppercase)
    char_to_idx = {ch: i for i, ch in enumerate(vocab)}
    vocab_size = len(vocab)
    n_samples, seq_len = char_arrays.shape
    one_hot = np.zeros((n_samples, seq_len, vocab_size), dtype=np.float32)
    for i, seq in enumerate(char_arrays):
        for t, ch in enumerate(seq):
            if ch in char_to_idx:
                one_hot[i, t, char_to_idx[ch]] = 1.0
    return one_hot

class VanillaRNN:
    def __init__(self, vocab_size, hidden_size=128, alpha=0.0001):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.alpha = alpha
        self.Wxh = np.random.randn(hidden_size, vocab_size) * 0.01
        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.Why = np.random.randn(vocab_size, hidden_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((vocab_size, 1))

    def feed_forward(self, x):
        if not hasattr(self, 'h'):
            self.h = np.zeros((self.hidden_size, 1))
        self.h = np.tanh(np.dot(self.Wxh, x) + np.dot(self.Whh, self.h) + self.bh)
        return type('RNNOutput', (), {'h': self.h, 'states': [self.h]})

    def train(self, inputs, targets, epochs=10):
        n_samples, seq_len, _ = inputs.shape
        for epoch in range(epochs):
            total_loss = 0.0
            dWxh = np.zeros_like(self.Wxh)
            dWhh = np.zeros_like(self.Whh)
            dWhy = np.zeros_like(self.Why)
            dbh = np.zeros_like(self.bh)
            dby = np.zeros_like(self.by)
            for i in range(n_samples):
                self.h = np.zeros((self.hidden_size, 1))
                hs = []
                ys = []
                for t in range(seq_len):
                    x = inputs[i, t].reshape(-1, 1)
                    out = self.feed_forward(x)
                    hs.append(out.h.copy())
                    y = np.dot(self.Why, out.h) + self.by
                    ys.append(y)
                loss = 0.0
                for t in range(seq_len):
                    y = ys[t]
                    exp_y = np.exp(y - np.max(y))
                    probs = exp_y / np.sum(exp_y)
                    target = targets[i, t].reshape(-1, 1)
                    loss += -np.log(probs[np.argmax(target)])
                total_loss += loss

                dWhy_t = np.zeros_like(self.Why)
                dby_t = np.zeros_like(self.by)
                dh_next = np.zeros((self.hidden_size, 1))
                for t in reversed(range(seq_len)):
                    y = ys[t]
                    exp_y = np.exp(y - np.max(y))
                    probs = exp_y / np.sum(exp_y)
                    target = targets[i, t].reshape(-1, 1)
                    dy = probs - target
                    dWhy_t += np.dot(dy, hs[t].T)
                    dby_t += dy
                    dh = np.dot(self.Why.T, dy) + dh_next
                    dh_raw = (1 - hs[t] ** 2) * dh
                    dWxh_t = np.dot(dh_raw, inputs[i, t].reshape(1, -1))
                    dWhh_t = np.dot(dh_raw, hs[t-1].T if t > 0 else np.zeros((self.hidden_size, 1)).T)
                    dbh_t = dh_raw
                    dWxh += dWxh_t
                    dWhh += dWhh_t
                    dbh += dbh_t
                    dh_next = np.dot(self.Whh.T, dh_raw)
                dWhy += dWhy_t
                dby += dby_t

            self.Wxh -= self.alpha * (dWxh / n_samples)
            self.Whh -= self.alpha * (dWhh / n_samples)
            self.Why -= self.alpha * (dWhy / n_samples)
            self.bh -= self.alpha * (dbh / n_samples)
            self.by -= self.alpha * (dby / n_samples)

            loss_value = (total_loss / n_samples).item()
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_value:.4f}")

if __name__ == "__main__":
    inputs = np.array([
        ["A","B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z"],
        ["Z","Y","X","W","V","U","T","S","R","Q","P","O","N","M","L","K","J","I","H","G","F","E","D","C","B","A"],
        ["B","D","F","H","J","L","N","P","R","T","V","X","Z","A","C","E","G","I","K","M","O","Q","S","U","W","Y"],
        ["M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z","A","B","C","D","E","F","G","H","I","J","K","L"],
        ["H","G","F","E","D","C","B","A","L","K","J","I","P","O","N","M","U","T","S","R","Q","X","W","V","Z","Y"]
    ])
    expected = np.array([
        ["B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z","A"],
        ["A","B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z"],
        ["C","E","G","I","K","M","O","Q","S","U","W","Y","A","B","D","F","H","J","L","N","P","R","T","V","X","Z"],
        ["N","O","P","Q","R","S","T","U","V","W","X","Y","Z","A","B","C","D","E","F","G","H","I","J","K","L","M"],
        ["I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z","A","B","C","D","E","F","G","H"]
    ])

    one_hot_inputs = string_to_one_hot(inputs)
    one_hot_expected = string_to_one_hot(expected)

    rnn = VanillaRNN(vocab_size=len(string.ascii_uppercase), hidden_size=128, alpha=0.0001)
    rnn.train(one_hot_inputs, one_hot_expected, epochs=10)

    new_inputs = np.array([["B", "C", "D"]])
    for input_vec in string_to_one_hot(new_inputs)[0]:
        predictions = rnn.feed_forward(input_vec.reshape(-1, 1))
        output_index = np.argmax(np.dot(rnn.Why, predictions.h) + rnn.by)
        print(output_index)
        print(string.ascii_uppercase[output_index])

Epoch 1/10, Loss: 84.7044
Epoch 2/10, Loss: 84.7043
Epoch 3/10, Loss: 84.7043
Epoch 4/10, Loss: 84.7043
Epoch 5/10, Loss: 84.7043
Epoch 6/10, Loss: 84.7042
Epoch 7/10, Loss: 84.7042
Epoch 8/10, Loss: 84.7042
Epoch 9/10, Loss: 84.7041
Epoch 10/10, Loss: 84.7041
24
Y
22
W
12
M
